## 1. Install Dependencies

In [ ]:
# Install required packages
# These cover all dependencies needed by model.py, dataset.py, and training

# Fix protobuf version conflict first
!pip uninstall -y protobuf
!pip install -q protobuf==3.20.3

# Install transformers and dependencies
!pip install -q transformers>=4.35.0 accelerate sentencepiece

# Install torch-geometric
!pip install -q huggingface-hub tqdm

# Note: torch, numpy, pandas, scikit-learn are pre-installed on Kaggle
print("✓ All dependencies installed")

## 2. Setup and Imports

In [ ]:
import os
import sys
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import get_linear_schedule_with_warmup
from tqdm import tqdm
from pathlib import Path
import json
import gc
import random
import numpy as np

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Check environment
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    torch.cuda.empty_cache()
    gc.collect()
    
# Check if running on Kaggle
IS_KAGGLE = os.path.exists('/kaggle')
print(f"Running on Kaggle: {IS_KAGGLE}")

set_seed(42)
print(f"✓ Random seed set to 42 for reproducibility")

PyTorch version: 2.9.1+cpu
CUDA available: False
Running on Kaggle: False


## 3. Configuration (Kaggle-Optimized)

In [ ]:
# ==============================================================================
# RESUME TRAINING CONFIGURATION
# ==============================================================================
# Set to True to resume from last checkpoint, False to train from scratch
RESUME_TRAINING = False  # Change to True to resume from checkpoint

# If resuming, specify the checkpoint directory (leave None to auto-detect)
RESUME_CHECKPOINT_DIR = None  # e.g., '/kaggle/input/previous-training/checkpoints'

# ==============================================================================
# DATA CONFIGURATION
# ==============================================================================
if IS_KAGGLE:
    DATA_CONFIG = {
        'train_jsonl': '/kaggle/input/gwm-e-cora-link/cora_train_link_data.jsonl',
        'val_jsonl': '/kaggle/input/gwm-e-cora-link/cora_val_link_data.jsonl',
        'test_jsonl': '/kaggle/input/gwm-e-cora-link/cora_test_link_data.jsonl',
        'train_embedding_path': '/kaggle/input/gwm-e-cora-link/train_edge_embeddings.pt',
        'val_embedding_path': '/kaggle/input/gwm-e-cora-link/val_edge_embeddings.pt',
        'test_embedding_path': '/kaggle/input/gwm-e-cora-link/test_edge_embeddings.pt',
    }
else:
    DATA_CONFIG = {
        'train_jsonl': 'data/cora/processed/link-prediction/cora_train_link_data.jsonl',
        'val_jsonl': 'data/cora/processed/link-prediction/cora_val_link_data.jsonl',
        'test_jsonl': 'data/cora/processed/link-prediction/cora_test_link_data.jsonl',
        'train_embedding_path': 'data/cora/processed/link-prediction/train_edge_embeddings.pt',
        'val_embedding_path': 'data/cora/processed/link-prediction/val_edge_embeddings.pt',
        'test_embedding_path': 'data/cora/processed/link-prediction/test_edge_embeddings.pt',
    }

# Model configuration - ADJUSTED FOR LINK PREDICTION
MODEL_CONFIG = {
    'llama_model': 'meta-llama/Llama-3.2-3B-Instruct',
    'graph_embedding_dim': 768,  # 768D
    'projector_hidden_dim': 3072,  # Slightly larger for link prediction complexity
    'num_hops': 4,  # 2 hops per node × 2 nodes = 4 total hops
}

# Training configuration - Optimized for link prediction (binary classification)
TRAINING_CONFIG = {
    'batch_size': 1,  # Reduced from 4 to save memory
    'gradient_accumulation_steps': 32,  # Increased to maintain effective batch = 32
    'learning_rate': 3e-5,  # Slightly higher for binary task
    'weight_decay': 0.1,
    'num_epochs': 10,
    'warmup_steps': 50,
    'num_workers': 0,  # Reduced from 2 to save memory (use 0 for single GPU)
    'output_dir': '/kaggle/working/checkpoints' if IS_KAGGLE else './trained_models/checkpoints',
    'save_every': 1,
    'use_fp16': True,
    'early_stopping_patience': 5,  # Faster early stopping for link prediction
    'dropout': 0.1,
    'max_grad_norm': 1.0,
}

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print("\n" + "="*60)
print(" "*10 + "GWM-E LINK PREDICTION TRAINING")
print("="*60)

if RESUME_TRAINING:
    print("\n🔄 MODE: RESUME FROM CHECKPOINT")
    if RESUME_CHECKPOINT_DIR:
        print(f"   Checkpoint dir: {RESUME_CHECKPOINT_DIR}")
    else:
        print(f"   Will auto-detect checkpoint in output_dir")
else:
    print("\n🆕 MODE: TRAIN FROM SCRATCH")

print("\n📊 Task: Link Prediction (Binary: yes/no)")
print(f"   Input: Source node (2 hops) + Target node (2 hops)")
print(f"   Embedding dim: {MODEL_CONFIG['graph_embedding_dim']}D (4 hops × 768D)")

print("\nData:")
for k, v in DATA_CONFIG.items():
    print(f"  {k}: {v}")
print("\nModel:")
for k, v in MODEL_CONFIG.items():
    print(f"  {k}: {v}")
print("\nTraining:")
for k, v in TRAINING_CONFIG.items():
    print(f"  {k}: {v}")
print(f"\nDevice: {device}")

print(f"Effective batch size: {TRAINING_CONFIG['batch_size'] * TRAINING_CONFIG['gradient_accumulation_steps']}")
print("="*60)


                    KAGGLE-OPTIMIZED CONFIGURATION

Data:
  train_jsonl: ../data/cora/cora_train_node_data.jsonl
  test_jsonl: ../data/cora/cora_test_node_data.jsonl
  embedding_path: ../data/cora/multi_hop_graph_embedding.pt

Model (Memory-Optimized):
  llama_model: meta-llama/Llama-3.2-3B-Instruct
  graph_embedding_dim: 2048
  projector_hidden_dim: 2048
  num_hops: 5
  use_8bit: True

Training (Kaggle-Optimized):
  batch_size: 2
  gradient_accumulation_steps: 16
  learning_rate: 0.0002
  num_epochs: 5
  warmup_steps: 50
  num_workers: 2
  output_dir: ./checkpoints
  save_every: 1
  use_fp16: True

Device: cpu
Effective batch size: 32


## 4. Copy Model Files

Copy the model architecture files to working directory.

In [ ]:
# Clone GitHub repo and copy model files to working directory
required_files = ['model.py', 'dataset.py', 'inference.py']

if IS_KAGGLE:
    print("="*70)
    print("Cloning GitHub repository...")
    print("="*70)
    
    # Clone your GitHub repo (UPDATE with your repo URL)
    GITHUB_REPO = "https://github.com/HiIamPhuc/GWM.git"
    
    # Clone repo
    !git clone {GITHUB_REPO} /kaggle/working/gwm
    
    # Copy model files from repo to working directory
    repo_path = Path("/kaggle/working/gwm/gwm/link-prediction")
    
    if repo_path.exists():
        print(f"\n✓ Repository cloned successfully")
        print(f"Source: {repo_path}")
        
        # Copy files
        for file in required_files:
            !cp {repo_path}/{file} /kaggle/working/
            print(f"✓ Copied {file}")

    else:
        print(f"\n❌ Repository path not found: {repo_path}")
        print("Please check the repository structure")
else:
    print("Running locally - files should be in current directory")

# Verify files exist
missing_files = [f for f in required_files if not os.path.exists(f)]

if missing_files:
    print(f"\n❌ Missing files: {missing_files}")
    raise FileNotFoundError(f"Required files not found: {missing_files}")
else:
    print(f"\n✓ All required files ready: {required_files}")

Running locally - files should be in current directory

✓ All required files ready:
  - model.py
  - dataset.py


## 5. Load Model with Quantization

In [ ]:
from kaggle_secrets import UserSecretsClient
HF_TOKEN = "HF_TOKEN"
HF_TOKEN = UserSecretsClient().get_secret(HF_TOKEN)
!hf auth login --token {HF_TOKEN}

In [ ]:
from model import GWM_E

print("="*60)
print(" "*15 + "Loading GWM-E Model")
print("="*60)

# Clear GPU memory before loading
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

# Initialize model with dropout for regularization
print(f"\nLoading {MODEL_CONFIG['llama_model']}...")

model = GWM_E(
    llama_model_path=MODEL_CONFIG['llama_model'],
    graph_embedding_dim=MODEL_CONFIG['graph_embedding_dim'],
    projector_hidden_dim=MODEL_CONFIG['projector_hidden_dim'],
    num_hops=MODEL_CONFIG['num_hops'],
    freeze_llm=True,
    dropout=TRAINING_CONFIG.get('dropout', 0.1),
)

# Calculate parameters
llm_params = sum(p.numel() for p in model.llm.parameters()) / 1e9
projector_params = sum(p.numel() for p in model.projector.parameters()) / 1e6
trainable_params = sum(p.numel() for p in model.projector.parameters() if p.requires_grad) / 1e6

print(f"\nModel Statistics:")
print(f"  LLaMA parameters: {llm_params:.2f}B (frozen)")
print(f"  Projector parameters: {projector_params:.2f}M (trainable)")
print(f"  Total trainable: {trainable_params:.2f}M")

# Check GPU memory
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    allocated = torch.cuda.memory_allocated(0) / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"\nGPU Memory: {allocated:.2f} GB / {total:.2f} GB")

print("\n✓ Model loaded successfully!")
print("="*60)


                    Loading GWM-E Model

Loading meta-llama/Llama-3.2-3B-Instruct...
Using 8-bit quantization to fit in 16GB GPU...
This may take 2-3 minutes...



`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

: 

## 6. Load Datasets

In [ ]:
from dataset import create_dataloaders, GWMDataset

print("="*70)
print(" "*20 + "Loading Datasets")
print("="*70)

print("\n📋 Note: Link prediction uses separate edge embeddings for each split")
print("   Each edge embedding combines source + target node multi-hop neighborhoods")

# Load train dataset with train edge embeddings
train_dataset = GWMDataset(
    jsonl_path=DATA_CONFIG['train_jsonl'],
    embedding_path=DATA_CONFIG['train_embedding_path'],
    tokenizer=model.tokenizer,
    num_hops=MODEL_CONFIG['num_hops'],
)

# Load validation dataset with validation edge embeddings
val_dataset = GWMDataset(
    jsonl_path=DATA_CONFIG['val_jsonl'],
    embedding_path=DATA_CONFIG['val_embedding_path'],
    tokenizer=model.tokenizer,
    num_hops=MODEL_CONFIG['num_hops'],
)

# Load test dataset with test edge embeddings
test_dataset = GWMDataset(
    jsonl_path=DATA_CONFIG['test_jsonl'],
    embedding_path=DATA_CONFIG['test_embedding_path'],
    tokenizer=model.tokenizer,
    num_hops=MODEL_CONFIG['num_hops'],
)

# Create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=TRAINING_CONFIG['batch_size'],
    shuffle=True,
    num_workers=TRAINING_CONFIG['num_workers'],
)

val_loader = DataLoader(
    val_dataset,
    batch_size=TRAINING_CONFIG['batch_size'],
    shuffle=False,
    num_workers=TRAINING_CONFIG['num_workers'],
)

print(f"\n✓ Datasets loaded successfully!")
print(f"  Training samples: {len(train_dataset):,}")
print(f"  Validation samples: {len(val_dataset):,}")
print(f"  Test samples: {len(test_dataset):,}")
print(f"  Training batches: {len(train_loader):,}")
print(f"  Validation batches: {len(val_loader):,}")

# Verify embedding shape
sample = train_dataset[0]
print(f"\n📐 Edge embedding shape: {sample['multi_hop_embedding'].shape}")
print(f"   Expected: [4, 768] (2 hops source + 2 hops target, 768D each)")
print(f"   Flattened for model: {sample['multi_hop_embedding'].flatten().shape[0]}D")

## 7. Setup Training

Configure optimizer, scheduler, and training components. Includes checkpoint resumption support.

In [ ]:
print("="*70)
print(" "*20 + "Setting up Training")
print("="*70)

# Create output directory
output_dir = Path(TRAINING_CONFIG['output_dir'])
output_dir.mkdir(parents=True, exist_ok=True)

# ==============================================================================
# CHECKPOINT RESUMPTION LOGIC
# ==============================================================================
resume_from_epoch = 0
loaded_history = []
best_accuracy = 0
best_epoch = 0
patience_counter = 0

if RESUME_TRAINING:
    # Determine checkpoint directory
    if RESUME_CHECKPOINT_DIR:
        checkpoint_dir = Path(RESUME_CHECKPOINT_DIR)
    else:
        checkpoint_dir = output_dir
    
    print(f"\n🔍 Checking for existing checkpoints in: {checkpoint_dir}")
    
    # Check for required files
    history_file = checkpoint_dir / "training_history.json"
    last_checkpoint = checkpoint_dir / "projector_last.pt"
    best_checkpoint = checkpoint_dir / "projector_best.pt"
    config_file = checkpoint_dir / "training_config.json"
    
    if history_file.exists() and last_checkpoint.exists():
        print(f"✓ Found checkpoint files!")
        
        # Load training history
        with open(history_file, 'r') as f:
            loaded_history = json.load(f)
        
        resume_from_epoch = loaded_history[-1]['epoch']
        print(f"  • Training history: {len(loaded_history)} epochs completed")
        print(f"  • Last epoch: {resume_from_epoch}")
        print(f"  • Last train loss: {loaded_history[-1]['train_loss']:.4f}")
        print(f"  • Last val accuracy: {loaded_history[-1]['val_accuracy']:.4f}")
        
        # Find best epoch from history
        best_entry = max(loaded_history, key=lambda x: x['val_accuracy'])
        best_accuracy = best_entry['val_accuracy']
        best_epoch = best_entry['epoch']
        print(f"  • Best validation accuracy: {best_accuracy:.4f} at epoch {best_epoch}")
        
        # Calculate patience counter
        epochs_since_best = resume_from_epoch - best_epoch
        patience_counter = epochs_since_best
        print(f"  • Patience counter: {patience_counter}/{TRAINING_CONFIG['early_stopping_patience']}")
        
        # Load last checkpoint weights
        print(f"\n📥 Loading model weights from: {last_checkpoint.name}")
        model.load_projector(str(last_checkpoint))
        print(f"✓ Model weights loaded successfully!")
        
        # Load previous configuration for reference
        if config_file.exists():
            with open(config_file, 'r') as f:
                previous_config = json.load(f)
            print(f"\n📋 Previous configuration loaded")
            
            # Warn about config changes
            config_changed = False
            for key in MODEL_CONFIG:
                if key in previous_config and previous_config[key] != MODEL_CONFIG[key]:
                    print(f"  ⚠️  {key}: {previous_config[key]} → {MODEL_CONFIG[key]}")
                    config_changed = True
            
            if config_changed:
                print(f"  ⚠️  WARNING: Configuration has changed! This may affect training.")
        
        print(f"\n🚀 Resuming training from epoch {resume_from_epoch + 1}")
        
    else:
        print(f"❌ Checkpoint files not found!")
        print(f"   Required files:")
        print(f"   • training_history.json: {'✓' if history_file.exists() else '✗'}")
        print(f"   • projector_last.pt: {'✓' if last_checkpoint.exists() else '✗'}")
        print(f"\n⚠️  Falling back to training from scratch...")
        RESUME_TRAINING = False

# ==============================================================================
# SAVE CONFIGURATION
# ==============================================================================
config_path = output_dir / "training_config.json"
all_config = {**DATA_CONFIG, **MODEL_CONFIG, **TRAINING_CONFIG}
with open(config_path, 'w') as f:
    json.dump(all_config, f, indent=2)
print(f"\n✓ Saved config to: {config_path}")

# ==============================================================================
# SETUP OPTIMIZER AND SCHEDULER
# ==============================================================================
# Optimizer with proper weight decay for regularization
optimizer = torch.optim.AdamW(
    model.projector.parameters(),
    lr=TRAINING_CONFIG['learning_rate'],
    weight_decay=TRAINING_CONFIG.get('weight_decay', 0.1),
)
print(f"✓ Optimizer: AdamW (lr={TRAINING_CONFIG['learning_rate']}, wd={TRAINING_CONFIG.get('weight_decay', 0.1)})")

# Scheduler - adjust for resumed training
remaining_epochs = TRAINING_CONFIG['num_epochs'] - resume_from_epoch
total_steps = len(train_loader) * remaining_epochs // TRAINING_CONFIG['gradient_accumulation_steps']
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=TRAINING_CONFIG['warmup_steps'] if not RESUME_TRAINING else 0,
    num_training_steps=total_steps,
)
print(f"✓ Scheduler: Linear warmup + decay")
print(f"  Remaining epochs: {remaining_epochs}")
print(f"  Total steps: {total_steps:,}")
if not RESUME_TRAINING:
    print(f"  Warmup steps: {TRAINING_CONFIG['warmup_steps']:,}")
else:
    print(f"  Warmup steps: 0 (skipped for resumed training)")

# Mixed precision (optional)
scaler = None
if TRAINING_CONFIG.get('use_fp16', False) and torch.cuda.is_available():
    scaler = torch.cuda.amp.GradScaler()
    print("✓ Mixed precision (FP16) enabled")

print("\n✓ Training setup complete!")
if RESUME_TRAINING:
    print(f"✓ Ready to resume from epoch {resume_from_epoch + 1}")
else:
    print(f"✓ Ready to train from scratch")

## 8. Training Functions

In [ ]:
# Import evaluation functions from inference.py
from inference import generate_predictions, evaluate_predictions


def train_epoch(model, train_loader, optimizer, scheduler, epoch, gradient_accumulation_steps, device, scaler=None, max_grad_norm=1.0, verbose=True):
    model.train()
    total_loss = 0
    num_batches = 0
    nan_count = 0
    
    # Only show progress bar if verbose=True
    if verbose:
        progress_bar = tqdm(train_loader, desc=f"Epoch {epoch}", ncols=100)
    else:
        progress_bar = train_loader
    
    for batch_idx, batch in enumerate(progress_bar):
        multi_hop_embedding = batch['multi_hop_embedding'].to(device)
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        # Mixed precision forward pass
        if scaler is not None:
            with torch.amp.autocast('cuda'):
                logits, loss = model(
                    multi_hop_embeddings=multi_hop_embedding,
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    labels=labels,
                )

            # Check for NaN loss
            if torch.isnan(loss) or torch.isinf(loss):
                print(f"\n⚠️  WARNING: NaN/Inf loss detected at batch {batch_idx}, skipping...")
                nan_count += 1
                continue
            
            loss = loss / gradient_accumulation_steps
            scaler.scale(loss).backward()
        else:
            logits, loss = model(
                multi_hop_embeddings=multi_hop_embedding,
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            
            # Check for NaN loss
            if torch.isnan(loss) or torch.isinf(loss):
                print(f"\n⚠️  WARNING: NaN/Inf loss detected at batch {batch_idx}, skipping...")
                nan_count += 1
                continue
            
            loss = loss / gradient_accumulation_steps
            loss.backward()
        
        # Update weights
        if (batch_idx + 1) % gradient_accumulation_steps == 0:
            if scaler is not None:
                scaler.unscale_(optimizer)
                # Clip gradients to prevent explosion
                torch.nn.utils.clip_grad_norm_(model.projector.parameters(), max_norm=max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
            else:
                # Clip gradients to prevent explosion
                torch.nn.utils.clip_grad_norm_(model.projector.parameters(), max_norm=max_grad_norm)
                optimizer.step()
            
            scheduler.step()
            optimizer.zero_grad()
        
        total_loss += loss.item() * gradient_accumulation_steps
        num_batches += 1
        
        # Only update progress bar if verbose
        if verbose and hasattr(progress_bar, 'set_postfix'):
            progress_bar.set_postfix({
                'loss': f"{loss.item() * gradient_accumulation_steps:.4f}",
                'lr': f"{scheduler.get_last_lr()[0]:.2e}"
            })
    
    if nan_count > 0:
        print(f"\n⚠️  Skipped {nan_count} batches due to NaN/Inf loss")
    
    avg_loss = total_loss / num_batches if num_batches > 0 else float('nan')
    return avg_loss


def evaluate(model, test_dataset, device, max_new_tokens=50, temperature=0.1, verbose=True):
    """
    Evaluate model using inference.py functions.
    Returns accuracy and predictions based on generated text comparison.
    
    Args:
        verbose: If False, suppresses the tqdm progress bar during evaluation
    """
    # Temporarily disable tqdm in generate_predictions by patching it
    import sys
    from io import StringIO
    
    if not verbose:
        # Redirect stdout to suppress tqdm output
        old_stdout = sys.stdout
        sys.stdout = StringIO()
    
    try:
        # Generate predictions using inference.py function
        predictions = generate_predictions(
            model=model,
            test_dataset=test_dataset,
            device=device,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            verbose=verbose,
        )
    finally:
        if not verbose:
            # Restore stdout
            sys.stdout = old_stdout
    
    # Evaluate predictions using inference.py function
    metrics = evaluate_predictions(predictions)
    
    # For consistency with training loop, return (accuracy, predictions)
    return metrics['accuracy'], predictions


print("✓ Training and evaluation functions ready")

## 9. Training Loop

In [ ]:
print("="*70)
print(" "*20 + "STARTING TRAINING")
print("="*70)

# Initialize from loaded values if resuming
training_history = loaded_history.copy() if RESUME_TRAINING else []
early_stopping_patience = TRAINING_CONFIG.get('early_stopping_patience', 5)

# Determine epoch range
start_epoch = resume_from_epoch + 1 if RESUME_TRAINING else 1
end_epoch = TRAINING_CONFIG['num_epochs']

if RESUME_TRAINING:
    print(f"\n📌 Resuming from epoch {start_epoch}/{end_epoch}")
    print(f"   Best so far: {best_accuracy:.4f} at epoch {best_epoch}")
    print(f"   Patience: {patience_counter}/{early_stopping_patience}")
else:
    print(f"\n📌 Training from scratch: epochs 1-{end_epoch}")

for epoch in range(start_epoch, end_epoch + 1):
    print(f"\n{'='*70}")
    print(f"Epoch {epoch}/{end_epoch}")
    print(f"{'='*70}")
    
    # Train
    train_loss = train_epoch(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        scheduler=scheduler,
        epoch=epoch,
        gradient_accumulation_steps=TRAINING_CONFIG['gradient_accumulation_steps'],
        device=device,
        scaler=scaler,
        max_grad_norm=TRAINING_CONFIG.get('max_grad_norm', 1.0),
        verbose=False,
    )
    
    # Evaluate using inference.py functions
    val_accuracy, predictions = evaluate(
        model=model,
        test_dataset=val_loader.dataset,
        device=device,
        max_new_tokens=50,
        temperature=0.1,
        verbose=False,
    )
    
    # Save predictions for this epoch
    predictions_path = output_dir / "predictions_last.json"
    with open(predictions_path, 'w', encoding='utf-8') as f:
        json.dump(predictions, f, indent=2, ensure_ascii=False)
    print(f"  ✓ Saved predictions of epoch {epoch}: {predictions_path.name}")
    
    # Compact logging - single line summary
    print(f"Results: Train Loss={train_loss:.4f} | Val Acc={val_accuracy:.4f} ({val_accuracy*100:.2f}%)")
    
    # Save history
    training_history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_accuracy': val_accuracy,
    })
    
    # Save history after each epoch (in case training is interrupted)
    history_path = output_dir / "training_history.json"
    with open(history_path, 'w') as f:
        json.dump(training_history, f, indent=2)
    print(f"  ✓ Saved training history: {history_path.name}")
    
    # Save checkpoint
    checkpoint_path = output_dir / f"projector_last.pt"
    model.save_projector(str(checkpoint_path))
    print(f"  ✓ Saved checkpoint of epoch {epoch}: {checkpoint_path.name}")
    
    # Track best model and early stopping
    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_epoch = epoch
        patience_counter = 0
        best_path = output_dir / "projector_best.pt"
        model.save_projector(str(best_path))
        print(f"  ⭐ New best: {best_accuracy:.4f} ({best_accuracy*100:.2f}%)")
        
        # Save best predictions as well
        best_predictions_path = output_dir / "predictions_best.json"
        with open(best_predictions_path, 'w', encoding='utf-8') as f:
            json.dump(predictions, f, indent=2, ensure_ascii=False)
        print(f"  ✓ Saved best predictions: {best_predictions_path.name}")
    else:
        patience_counter += 1
        print(f"  No improvement (patience: {patience_counter}/{early_stopping_patience})")
    
    # Clear memory after each epoch
    torch.cuda.empty_cache()
    gc.collect()
    
    # Early stopping check
    if patience_counter >= early_stopping_patience:
        print(f"\n{'='*70}")
        print(f"🛑 Early stopping triggered!")
        print(f"   No improvement for {early_stopping_patience} epochs")
        print(f"   Best validation accuracy: {best_accuracy:.4f} at epoch {best_epoch}")
        print(f"{'='*70}")
        break


history_path = output_dir / "training_history.json"
with open(history_path, 'w') as f:
    json.dump(training_history, f, indent=2)

print("\n" + "="*70)
print(" "*20 + "✅ TRAINING COMPLETED!")
print("="*70)
print(f"Best validation accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%) at epoch {best_epoch}")
print(f"Total epochs: {len(training_history)}")
print(f"Checkpoints: {output_dir}")

# Now evaluate on TEST set with best model
print("\n" + "="*70)
print(" "*20 + "FINAL TEST SET EVALUATION")
print("="*70)
print("Loading best model checkpoint...")

# Load best model
best_checkpoint = output_dir / "projector_best.pt"
model.load_projector(str(best_checkpoint))
print(f"✓ Loaded best model from epoch {best_epoch}")

# Evaluate on test set
print("\nEvaluating on test set...")
test_accuracy, test_predictions = evaluate(
    model=model,
    test_dataset=test_dataset,
    device=device,
    max_new_tokens=50,
    temperature=0.1,
    verbose=False,
)

# Save test predictions
test_predictions_path = output_dir / "predictions_test_final.json"
with open(test_predictions_path, 'w', encoding='utf-8') as f:
    json.dump(test_predictions, f, indent=2, ensure_ascii=False)

print("\n" + "="*70)
print(" "*20 + "FINAL RESULTS")
print("="*70)
print(f"Best Validation Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%) at epoch {best_epoch}")
print(f"Final Test Accuracy:      {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print("="*70)

# Save final summary
final_results = {
    "best_val_accuracy": best_accuracy,
    "best_epoch": best_epoch,
    "test_accuracy": test_accuracy,
    "total_epochs": len(training_history),
    "resumed_from_epoch": resume_from_epoch if RESUME_TRAINING else 0,
}
results_path = output_dir / "final_results.json"
with open(results_path, 'w') as f:
    json.dump(final_results, f, indent=2)
print(f"\n✓ Saved final results: {results_path.name}")

## 10. Visualize Training Progress

In [ ]:
import matplotlib.pyplot as plt

# Extract metrics
epochs = [h['epoch'] for h in training_history]
train_losses = [h['train_loss'] for h in training_history]
val_accuracies = [h['val_accuracy'] for h in training_history]

# Create plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
ax1.plot(epochs, train_losses, 'b-o', label='Train Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Accuracy plot
ax2.plot(epochs, [acc * 100 for acc in val_accuracies], 'g-o', label='Validation Accuracy')
ax2.axhline(y=test_accuracy * 100, color='r', linestyle='--', label=f'Test Accuracy: {test_accuracy*100:.2f}%')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Validation Accuracy (Text Generation)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"✓ Saved training curves to: {output_dir / 'training_curves.png'}")

## 11. Summary and Download

In [ ]:
print("="*70)
print(" "*25 + "TRAINING SUMMARY")
print("="*70)

print(f"\n📊 Results:")
print(f"  Best Validation Accuracy: {best_accuracy:.4f} ({best_accuracy*100:.2f}%) at epoch {best_epoch}")
print(f"  Final Test Accuracy:      {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"  Final Train Loss:         {training_history[-1]['train_loss']:.4f}")

print(f"\n💾 Saved Files:")
for file in sorted(output_dir.glob("*")):
    size = os.path.getsize(file) / (1024**2)
    print(f"  • {file.name} ({size:.1f} MB)")

if IS_KAGGLE:
    print(f"\n📥 Download from Kaggle:")
    print(f"  1. Go to 'Output' tab (right sidebar)")
    print(f"  2. Download 'checkpoints/' folder")
    print(f"  3. Key file: projector_best.pt (~{os.path.getsize(output_dir / 'projector_best.pt') / (1024**2):.1f} MB)")

print(f"\n🚀 Next Steps:")
print(f"  1. Download projector_best.pt")
print(f"  2. Use with LLaMA model for inference")
print(f"  3. Analyze predictions in predictions_test_final.json")

print("\n" + "="*70)

## Optional: Test a Single Prediction

Quick test to see model predictions.

In [ ]:
# Get a sample from test set
test_sample = test_dataset[0]
test_conv = test_dataset.conversations[0]

print("Testing model on a single example from TEST set...\n")
print("="*70)
print("Sample Input:")
print("="*70)
for turn in test_conv['conversations']:
    role = turn['from']
    message = turn['value'][:200] + "..." if len(turn['value']) > 200 else turn['value']
    print(f"{role.upper()}: {message}\n")

# Prepare inputs
multi_hop_emb = test_sample['multi_hop_embedding'].unsqueeze(0).to(device)
input_ids = test_sample['input_ids'].unsqueeze(0).to(device)
attention_mask = test_sample['attention_mask'].unsqueeze(0).to(device)

# Generate prediction
model.eval()
with torch.no_grad():
    outputs = model.generate(
        multi_hop_embeddings=multi_hop_emb,
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=50,
        temperature=0.1,
    )

# Decode
input_length = input_ids.shape[1]
generated = model.tokenizer.decode(outputs[0][input_length:], skip_special_tokens=True)

print("="*70)
print("Model Prediction:")
print("="*70)
print(f"ASSISTANT: {generated}\n")
print("="*70)